# 02 · Entendimento dos dados (EDA)

Fase 2 do CRISP-DM. Responde às perguntas analíticas 1, 2, 3, 5 e 6 do relatório das Etapas 1 e 2 sobre a tabela analítica (`dados/tabela_analitica.csv`, gerada pelo notebook 01).

- Unidade: escola × etapa (rede pública, matrícula ≥ 10 na etapa).
- Alvo de referência: taxa de abandono ≥ P80 da etapa (anos iniciais: taxa > 0, pois o P80 é zero).
- Associações não implicam causalidade. A geografia (pergunta 8) está no notebook 04, com o cálculo proporcional.
- A pergunta 7 (Ideb-aprendizagem) não foi executada: não há coluna de aprendizagem validada contra vazamento temporal.

In [ ]:
import os

REPO = "datasync-Projeto-CRISP-DM-abandono"
if not os.path.exists(REPO) and os.path.basename(os.getcwd()) != REPO:
    !git clone -q https://github.com/LuisGomes25/{REPO}.git
if os.path.basename(os.getcwd()) != REPO:
    os.chdir(REPO)

SAIDA = "resultados/02_eda"
os.makedirs(SAIDA, exist_ok=True)
print("Pasta de trabalho:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ETAPAS = ["ensino_medio", "anos_finais", "anos_iniciais"]

GRUPOS_VARIAVEIS = {
    "estruturais": ["indice_infra", "pct_transporte_publico", "pct_integral", "pct_noturno"],
    "docentes": ["afd_formacao_adequada", "dsu_doc_superior", "ied_esforco_alto", "ird_regularidade"],
    "pedagogicas": ["atu_alunos_turma", "had_horas_aula"],
    "gestao": ["icg_complexidade"],
    "contextuais": ["inse_media", "tdi_distorcao", "log_matriculas", "alunos_por_docente"],
}
NUMERICAS = [v for vs in GRUPOS_VARIAVEIS.values() for v in vs]
CATEGORICAS = ["SG_UF", "NO_CATEGORIA", "NO_DEPENDENCIA"]

df = pd.read_csv("dados/tabela_analitica.csv", encoding="utf-8-sig",
                 dtype={"codigo_escola": str, "CO_MUNICIPIO": str}, low_memory=False)
assert df.shape == (136713, 27), df.shape

def criar_alvo(serie, percentil=80):
    """Alto abandono por percentil da etapa; se o limite for zero, usa taxa > 0."""
    limite = float(serie.quantile(percentil / 100))
    if limite == 0.0:
        return (serie > 0).astype(int), limite
    return (serie >= limite).astype(int), limite

df["alvo_p80"] = 0
for etapa in ETAPAS:
    mask = df["etapa"] == etapa
    df.loc[mask, "alvo_p80"] = criar_alvo(df.loc[mask, "taxa_abandono"])[0].values

def salvar(tabela, nome, **kw):
    tabela.to_csv(os.path.join(SAIDA, nome), encoding="utf-8-sig", **kw)

print(df.groupby("etapa")["alvo_p80"].agg(["size", "mean"]).round(4))

## Pergunta 1 · Distribuição do abandono por etapa, rede, localização e UF

In [ ]:
def resumo(grupo):
    return grupo["taxa_abandono"].agg(
        linhas="size", media="mean", mediana="median", desvio="std",
        p75=lambda s: s.quantile(.75), p80=lambda s: s.quantile(.80), p90=lambda s: s.quantile(.90),
        pct_zero=lambda s: (s == 0).mean() * 100,
    ).round(3)

por_etapa = resumo(df.groupby("etapa"))
por_rede = resumo(df.groupby(["etapa", "NO_DEPENDENCIA"]))
por_localizacao = resumo(df.groupby(["etapa", "NO_CATEGORIA"]))
por_uf = resumo(df.groupby(["etapa", "SG_UF"]))

for tabela, nome in [(por_etapa, "01_por_etapa.csv"), (por_rede, "02_por_rede.csv"),
                     (por_localizacao, "03_por_localizacao.csv"), (por_uf, "04_por_uf.csv")]:
    salvar(tabela, nome)
display(por_etapa, por_rede, por_localizacao)

In [ ]:
fig, eixo = plt.subplots(figsize=(9, 5))
eixo.boxplot([df.loc[df["etapa"] == e, "taxa_abandono"] for e in ETAPAS], showfliers=False)
eixo.set_xticks([1, 2, 3], [e.replace("_", " ") for e in ETAPAS])
eixo.set_ylabel("Taxa de abandono (%)")
eixo.set_title("Distribuição da taxa de abandono por etapa (sem outliers)")
plt.savefig(os.path.join(SAIDA, "01_boxplot_etapa.png"), dpi=300, bbox_inches="tight")
plt.show()

## Pergunta 2 · Fatores associados ao alto abandono

In [ ]:
from scipy.stats import spearmanr, pointbiserialr, chi2_contingency

linhas = []
for etapa in ETAPAS:
    d = df[df["etapa"] == etapa]
    for grupo, variaveis in GRUPOS_VARIAVEIS.items():
        for var in variaveis:
            par = d[[var, "taxa_abandono", "alvo_p80"]].dropna()
            rho, p_rho = spearmanr(par[var], par["taxa_abandono"])
            r_pb, p_pb = pointbiserialr(par["alvo_p80"], par[var])
            linhas.append({"etapa": etapa, "grupo": grupo, "variavel": var, "n": len(par),
                           "spearman_taxa": rho, "p_spearman": p_rho,
                           "ponto_bisserial_alvo": r_pb, "p_ponto_bisserial": p_pb})
assoc = pd.DataFrame(linhas).round(4)
salvar(assoc, "05_associacoes_numericas.csv", index=False)

def cramers_v(x, y):
    tabela = pd.crosstab(x, y)
    chi2 = chi2_contingency(tabela)[0]
    n = tabela.values.sum()
    r, k = tabela.shape
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    r_c, k_c = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2 / min(k_c - 1, r_c - 1))

cramer = pd.DataFrame([{"etapa": e, "variavel": c,
                        "cramers_v": cramers_v(df.loc[df["etapa"] == e, c], df.loc[df["etapa"] == e, "alvo_p80"])}
                       for e in ETAPAS for c in CATEGORICAS]).round(3)
salvar(cramer, "06_cramers_v.csv", index=False)

display(assoc.sort_values("spearman_taxa", key=abs, ascending=False).groupby("etapa").head(5), cramer)

## Pergunta 3 · As associações persistem com INSE semelhante?

In [ ]:
import statsmodels.formula.api as smf

linhas = []
for etapa in ETAPAS:
    d = df.loc[df["etapa"] == etapa, ["taxa_abandono", "indice_infra", "inse_media"]].dropna()
    sem = smf.ols("taxa_abandono ~ indice_infra", data=d).fit(cov_type="HC3")
    com = smf.ols("taxa_abandono ~ indice_infra + inse_media", data=d).fit(cov_type="HC3")
    linhas.append({"etapa": etapa, "n": len(d),
                   "coef_infra_sem_inse": sem.params["indice_infra"], "p_sem": sem.pvalues["indice_infra"],
                   "coef_infra_com_inse": com.params["indice_infra"], "p_com": com.pvalues["indice_infra"],
                   "coef_inse": com.params["inse_media"], "r2_sem": sem.rsquared, "r2_com": com.rsquared})
controle_inse = pd.DataFrame(linhas).round(4)
salvar(controle_inse, "07_controle_inse.csv", index=False)

d = df.dropna(subset=["inse_media"]).copy()
d["quintil_inse"] = d.groupby("etapa")["inse_media"].transform(
    lambda s: pd.qcut(s, 5, labels=["Q1 (menor)", "Q2", "Q3", "Q4", "Q5 (maior)"]))
quintis = (d.groupby(["etapa", "quintil_inse"], observed=True)
             .agg(escolas=("alvo_p80", "size"),
                  pct_alto_abandono=("alvo_p80", lambda s: s.mean() * 100),
                  taxa_media=("taxa_abandono", "mean")).round(2))
salvar(quintis, "08_abandono_por_quintil_inse.csv")
display(controle_inse, quintis)

## Pergunta 5 · Perfis: escolas de alto abandono × demais

In [ ]:
from scipy.stats import mannwhitneyu

linhas = []
for etapa in ETAPAS:
    d = df[df["etapa"] == etapa]
    for var in NUMERICAS:
        alto = d.loc[d["alvo_p80"] == 1, var].dropna()
        baixo = d.loc[d["alvo_p80"] == 0, var].dropna()
        teste = mannwhitneyu(alto, baixo, alternative="two-sided")
        linhas.append({"etapa": etapa, "variavel": var,
                       "mediana_alto": alto.median(), "mediana_demais": baixo.median(),
                       "rank_biserial": 2 * teste.statistic / (len(alto) * len(baixo)) - 1,
                       "p_valor": teste.pvalue})
perfis = pd.DataFrame(linhas).round(4)
salvar(perfis, "09_perfis_mann_whitney.csv", index=False)
perfis.sort_values("rank_biserial", key=abs, ascending=False).groupby("etapa").head(5)

## Pergunta 6 · Limiares P75, P80 e P90

In [ ]:
linhas = []
for etapa in ETAPAS:
    taxa = df.loc[df["etapa"] == etapa, "taxa_abandono"]
    for p in [75, 80, 90]:
        alvo, limite = criar_alvo(taxa, p)
        linhas.append({"etapa": etapa, "percentil": p, "limite": limite,
                       "escolas_alto": int(alvo.sum()), "pct_alto": round(alvo.mean() * 100, 2)})
limiares = pd.DataFrame(linhas)
salvar(limiares, "10_limiares.csv", index=False)

fig, eixo = plt.subplots(figsize=(8, 4.5))
for etapa in ETAPAS:
    d = limiares[limiares["etapa"] == etapa]
    eixo.plot(d["percentil"], d["pct_alto"], marker="o", label=etapa.replace("_", " "))
eixo.set_xticks([75, 80, 90])
eixo.set_xlabel("Percentil de corte")
eixo.set_ylabel("% de escolas de alto abandono")
eixo.legend()
plt.savefig(os.path.join(SAIDA, "02_limiares.png"), dpi=300, bbox_inches="tight")
plt.show()
limiares

## Multicolinearidade, correlações, distribuições e ausência

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

m = df[NUMERICAS].fillna(df[NUMERICAS].median())
m = m.sample(20000, random_state=SEED)
m = (m - m.mean()) / m.std()
vif = pd.DataFrame({"variavel": NUMERICAS,
                    "VIF": [variance_inflation_factor(m.values, i) for i in range(m.shape[1])]})
vif = vif.sort_values("VIF", ascending=False).round(2)
salvar(vif, "11_vif.csv", index=False)
vif

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(20, 6.5))
for i, etapa in enumerate(ETAPAS):
    corr = df.loc[df["etapa"] == etapa, NUMERICAS + ["taxa_abandono"]].corr(method="spearman")
    im = eixos[i].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    eixos[i].set_xticks(range(len(corr)))
    eixos[i].set_xticklabels(corr.columns, rotation=90, fontsize=7)
    eixos[i].set_yticks(range(len(corr)))
    eixos[i].set_yticklabels(corr.columns if i == 0 else [], fontsize=7)
    for a in range(len(corr)):
        for b in range(len(corr)):
            v = corr.iloc[a, b]
            eixos[i].text(b, a, f"{v:.2f}", ha="center", va="center", fontsize=5,
                          color="white" if abs(v) > 0.6 else "black")
    eixos[i].set_title(etapa.replace("_", " ").title(), fontweight="bold")
fig.colorbar(im, ax=eixos, shrink=0.7, label="Spearman")
plt.savefig(os.path.join(SAIDA, "03_matriz_correlacao.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, eixos = plt.subplots(4, 4, figsize=(16, 13))
eixos = eixos.ravel()
for i, var in enumerate(NUMERICAS):
    for etapa in ETAPAS:
        v = df.loc[df["etapa"] == etapa, var].dropna()
        eixos[i].hist(v[v <= v.quantile(0.99)], bins=40, histtype="step", density=True,
                      linewidth=1.3, label=etapa.replace("_", " "))
    eixos[i].set_title(var, fontsize=9)
    eixos[i].tick_params(labelsize=7)
for j in range(len(NUMERICAS), len(eixos)):
    eixos[j].axis("off")
eixos[0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(SAIDA, "04_histogramas_preditoras.png"), dpi=300, bbox_inches="tight")
plt.show()

ausencia = df.groupby("etapa")[NUMERICAS].apply(lambda d: d.isna().mean().mul(100).round(1)).T
salvar(ausencia, "12_ausencia_por_etapa.csv")
ausencia